# Sleep Onset V2 — fresh dual-path training
This notebook initializes V2 from scratch, trains on the official split, produces diagnostics, and packages a Codabench submission. It never loads V1 weights.

In [ ]:
from pathlib import Path
import json, os, shutil, sys, zipfile

ON_KAGGLE = Path('/kaggle').exists()
ON_COLAB = 'google.colab' in sys.modules
WORK_DIR = Path('/kaggle/working' if ON_KAGGLE else '/content' if ON_COLAB else Path.cwd())

# The source may be beside this notebook, attached as a Kaggle Dataset,
# or supplied as sleep-onset-v2-source.zip.
candidates = [Path.cwd(), Path.cwd().parent]
if ON_KAGGLE:
    candidates += [p.parent for p in Path('/kaggle/input').glob('**/config.yaml')]
PROJECT_DIR = next((p for p in candidates if (p / 'train.py').exists() and (p / 'features.py').exists()), None)
if PROJECT_DIR is None:
    archives = list(Path('/kaggle/input' if ON_KAGGLE else WORK_DIR).glob('**/sleep-onset-v2-source.zip'))
    if not archives:
        raise FileNotFoundError('Attach the V2 source Dataset or upload sleep-onset-v2-source.zip')
    PROJECT_DIR = WORK_DIR / 'sleep-onset-v2-source'
    with zipfile.ZipFile(archives[0]) as archive:
        archive.extractall(PROJECT_DIR)

DATA_DIR = WORK_DIR / 'neural_compet'
OUTPUT_DIR = WORK_DIR / 'sleep-v2'
print({'project': str(PROJECT_DIR), 'data': str(DATA_DIR), 'output': str(OUTPUT_DIR)})

In [ ]:
import subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT_DIR / 'requirements.txt')], check=True)

## Synthetic tests and architecture smoke test

In [ ]:
TEST_DIR = PROJECT_DIR / 'tests'
if TEST_DIR.exists():
    subprocess.run([sys.executable, '-m', 'pytest', '-q', '-p', 'no:cacheprovider', str(TEST_DIR)], cwd=PROJECT_DIR, check=True)
else:
    print('Cloud tests folder not attached; skipping pytest. The local V2 suite passed before upload.')
sys.path.insert(0, str(PROJECT_DIR))
import torch
from model import DualPathSleepNet

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
smoke = DualPathSleepNet().to(device)
x = torch.randn(2, 3, 4, 640, device=device)
prediction, state = smoke(x)
fused, values = smoke.window_embeddings(x[:, 0], return_diagnostics=True)
prediction.mean().backward()
assert prediction.shape == (2, 3) and fused.shape == (2, 128)
assert values['cnn'].shape == (2, 128) and values['spectral'].shape == (2, 64)
assert torch.isfinite(prediction).all()
print('device:', device)
print('parameters:', sum(p.numel() for p in smoke.parameters() if p.requires_grad))
print({name: tuple(value.shape) for name, value in values.items()})
del smoke, x, prediction, state, fused, values
if torch.cuda.is_available(): torch.cuda.empty_cache()

## Train, stream-validate, diagnose and package
The first run downloads about 1.1 GB. If the V1 session cache still exists under `DATA_DIR`, NeuralBench reuses it.

In [ ]:
command = [
    sys.executable, str(PROJECT_DIR / 'train.py'),
    '--data-dir', str(DATA_DIR),
    '--output-dir', str(OUTPUT_DIR),
    '--config', str(PROJECT_DIR / 'config.yaml'),
    '--download',
]
# To continue an interrupted run, append '--resume' after confirming last_model.pt exists.
subprocess.run(command, cwd=PROJECT_DIR, check=True)

In [ ]:
submission = OUTPUT_DIR / 'submission.zip'
diagnostics = OUTPUT_DIR / 'v2_diagnostics.zip'
with zipfile.ZipFile(submission) as archive:
    assert sorted(archive.namelist()) == ['model_weights.pt', 'submission.py']
print((OUTPUT_DIR / 'experiment_summary.md').read_text())
print('Download for Codabench:', submission)
print('Download for analysis:', diagnostics)

Optional Colab persistence: copy `OUTPUT_DIR` to Google Drive after training. Do not upload `v2_diagnostics.zip` to Codabench; only upload `submission.zip`.